---
title: GeoPandas
editor:
  render-on-save: true
execute:
  enabled: true
  cache: true
  freeze: auto
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benchmark-urbanism/cityseer-examples/blob/gh-pages/class/4_geopandas.ipynb)

::: {.callout-note}

## Learning Objectives

- Create GeoDataFrames from scratch and from spatial data files
- Index, filter, and manipulate rows and columns in a GeoDataFrame
- Transform between coordinate reference systems
- Apply spatial operations like buffering and distance calculations
- Plot and export geospatial data

:::

In urban analysis, working with spatial data often means handling numerous geographic features simultaneously, such as buildings, streets, or administrative boundaries. While processing each feature individually is possible, it's considerably more efficient to manage and analyse them collectively within a structured format.

For general data analysis in Python, the `pandas` library is the standard tool. Pandas provides the `DataFrame`, a powerful table-like structure where rows represent observations (e.g., individual buildings) and columns store their attributes (e.g., height, area, land use). This structure is conceptually similar to attribute tables found in GIS software such as QGIS or ArcGIS.

[**GeoPandas**](https://geopandas.org/en/stable/) extends pandas by incorporating first-class support for geographic data. It introduces the `GeoDataFrame`, which is effectively a pandas `DataFrame` enhanced with a dedicated column for storing geometry objects (such as points, lines, or polygons). This integration allows you to combine tabular data analysis with spatial operations.

Benefits of using GeoPandas include:

- Storing geometric data alongside other attributes.
- Reading and writing various spatial file formats (Shapefile, GeoPackage, GeoJSON, etc.).
- Performing spatial operations (buffering, intersection, union, etc.).
- Handling Coordinate Reference Systems (CRS).
- Easy plotting of spatial data.

## From Scratch

Although you'll frequently load spatial data directly from files, knowing how to create a `GeoDataFrame` manually is beneficial. To do this, you typically need:

1. Tabular data (often a list of dictionaries).
2. Corresponding geometry objects.
3. A Coordinate Reference System (CRS) identifier.

Let's begin with a simple example.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

First, we'll define some data as a list of dictionaries. Each dictionary will represent a city, storing its name and approximate coordinates (longitude, latitude).

In [ ]:
data = [
    {"city": "Madrid", "x": -3.7038, "y": 40.4168},
    {"city": "Barcelona", "x": 2.1734, "y": 41.3851},
    {"city": "Valencia", "x": -0.3774, "y": 39.4699}
]

data

Next, we need to create the corresponding geometry objects. As these are coordinates, `Point` objects are suitable. We can generate a list of `Point` objects directly from our data.

In [ ]:
# Create Point geometries from the x and y columns
locations = []

for city in data:
    x = city["x"]
    y = city["y"]
    geom = Point(x, y)
    locations.append(geom)

locations

With the structured data and geometry prepared, the final component is the CRS. Given that the coordinates are longitude and latitude, the appropriate CRS is WGS 84, identified by the EPSG code 4326.

We can now construct the `GeoDataFrame`:

In [ ]:
gdf_cities = gpd.GeoDataFrame(
    data,  # The list of dictionaries
    geometry=locations,  # The Shapely Points
    crs=4326  # Coordinate Reference System (WGS 84)
)

gdf_cities

## Indexing

Similar to pandas DataFrames, GeoDataFrames possess an index. By default, this is a range of integers. You can assign a more meaningful index using one of the columns (e.g., the city name) with the `set_index()` method. This often simplifies data selection.

In [ ]:
gdf_cities = gdf_cities.set_index("city")

gdf_cities

You can access a row using its index. Pandas provides a special indexing method, `.loc[]`, which allows you to access rows by their index label. This is particularly useful when you've set a meaningful index, such as city names.

In [ ]:
gdf_cities.loc["Madrid"]

You can also specify a column name. For instance, to retrieve the latitude of Barcelona:

In [ ]:
gdf_cities.loc['Barcelona', "y"]

To retrieve all rows for a specific column, use the `:` operator. For example, to get all rows for the `y` column:

In [ ]:
gdf_cities.loc[:, "y"]

An alternative way to access columns is by using the column name directly:

In [ ]:
gdf_cities["y"]

A particularly powerful feature of GeoDataFrames is the ability to filter rows based on their properties. To do this, you create a boolean mask, which can then be used to exclude rows you are not interested in. For instance, if you want to filter the cities to include only those with a latitude greater than 40, you can proceed as follows:

In [ ]:
# Create a boolean mask
mask = gdf_cities["y"] > 40

mask

This mask is a pandas Series of boolean values, where `True` indicates that the condition (y > 40) is met, and `False` indicates it is not. You can now apply this mask to filter the rows in the original GeoDataFrame.

In [ ]:
# Use the mask to filter the rows
gdf_cities.loc[mask]

As shown, only two rows are returned because Valencia, the third city, has a latitude below 40.

## Geospatial

GeoDataFrames possess special properties stemming from their spatial nature.

GeoPandas formally designates a particular column as the geometry; this is typically done during the creation of the GeoDataFrame. The geometry column contains Shapely geometry objects, which can be points, lines, or polygons. This column is often called `geometry`, but this is not always the case.

In [ ]:
gdf_cities.geometry

Similarly, GeoPandas formally associates the geometry with a given CRS, accessible via the `.crs` attribute.

In [ ]:
gdf_cities.crs

GeoPandas conveniently integrates with `matplotlib` for straightforward plotting. Calling `.plot()` on a GeoDataFrame will render its geometries.

In [ ]:
gdf_cities.plot()

### Coordinate Transformations

::: {.callout-note}

#### Geographic vs. projected CRS

A **geographic CRS** (e.g., WGS 84 / EPSG:4326) stores coordinates as longitude and latitude in **degrees** on the Earth's curved surface. A **projected CRS** (e.g., EPSG:3035, UTM zones) projects coordinates onto a flat surface in **metres**.

You generally want to use a projected CRS for the type of work we're doing in this notebook. To find an appropriate projected CRS for your study area, search for your city or country at [epsg.io](https://epsg.io). Common choices include UTM zones (accurate for local areas), EPSG:3035 (Europe-wide), and national projections like EPSG:27700 (Great Britain).
:::

GeoPandas makes CRS transformation very easy with the `to_crs()` method. Let's convert our cities GeoDataFrame to EPSG:3035, a projected equal-area CRS suitable for Europe (ETRS89-LAEA).

In [ ]:
gdf_cities_proj = gdf_cities.to_crs(3035)

gdf_cities_proj

Notice how the `x` and `y` columns no longer match the geometry, as the geometry is now in metres according to EPSG:3035. The `.crs` attribute reflects this change:

In [ ]:
gdf_cities_proj.crs

Plotting the transformed data shows the locations relative to the new projected coordinate system.

In [ ]:
gdf_cities_proj.plot()

### Spatial Operations

GeoPandas enables you to apply spatial operations directly to `GeoDataFrames` or `GeoSeries`. These operations are applied element-wise.

For instance, to create a 100km buffer around each city:

In [ ]:
# The buffer distance is in the units of the CRS (metres for EPSG:3035)
gdf_cities_proj["geometry"] = gdf_cities_proj.geometry.buffer(100000)  # 100km buffer

gdf_cities_proj.plot()

Alternatively, you can calculate the distance from each feature in a GeoDataFrame to a single Shapely geometry object. Let's determine the distance from each city buffer to a specific point (also defined in EPSG:3035 coordinates):

In [ ]:
# Define a point in the same CRS (EPSG:3035)
pt = Point(3300000, 2000000)

# Calculate distance from each geometry in the GeoSeries to the point
gdf_cities_proj.geometry.distance(pt)

### Creating and Updating Columns

If we wished to create a new column named 'easting' and assign it the x-coordinate of the geometry, we could proceed as follows:

In [ ]:
gdf_cities_proj['easting'] = gdf_cities_proj.geometry.centroid.x

gdf_cities_proj

Each geometry row in the DataFrame is a Shapely geometry, so you can use standard Shapely syntax to access properties such as the centroid or its x-coordinate.

If you decide instead to update the existing `x` and `y` columns, you can reference them directly:

In [ ]:
gdf_cities_proj['x'] = gdf_cities_proj.geometry.centroid.x
gdf_cities_proj['y'] = gdf_cities_proj.geometry.centroid.y

gdf_cities_proj

DataFrames offer an extensive variety of useful methods; here, we will use the `drop` method to remove the column we no longer wish to retain:

In [ ]:
gdf_cities_proj = gdf_cities_proj.drop(columns=['easting'])

gdf_cities_proj

## Working with Data

### Reading Data

GeoPandas can read various spatial data formats, such as Shapefiles, GeoPackages, and GeoJSON files. The `read_file()` function is the primary method for this. Note that the file path in the example below must be adjusted to point to the actual location of the file on your computer.

In [ ]:
mad_bldgs = gpd.read_file('../data/madrid_buildings/madrid_bldgs.gpkg')

mad_bldgs.head()

The `head()` method shows the first few rows of the GeoDataFrame, including the geometry column. The CRS is automatically detected and set.

### Plotting

Let's plot the data.

In [ ]:
mad_bldgs.plot()

To zoom in when plotting, you can set your x and y-axis limits. For a cleaner plot, it is also generally preferable to turn off the axes so that the coordinates do not render.

In [ ]:
ax = mad_bldgs.plot()

ax.set_xlim(439000, 442000)
ax.set_ylim(4473000, 4476000)
ax.axis('off')

Let's create a new column named `area`, which we will set to the value of each geometry's area. Then, we will plot the data again, this time rendering the colour according to the building's area.

In [ ]:
mad_bldgs['area'] = mad_bldgs.geometry.area

ax = mad_bldgs.plot(
    column='area',
    cmap='viridis',
    vmax=10000,
)

ax.set_xlim(439000, 442000)
ax.set_ylim(4473000, 4476000)
ax.axis('off')

### Saving Data

GeoPandas can save GeoDataFrames to various formats, including Shapefiles, GeoPackages, and GeoJSON files. The `to_file()` method is used for this purpose. You can specify the format using the `driver` parameter.

```python
mad_bldgs.to_file('bldgs_w_area.gpkg', driver='GPKG')
```

### Exercises

- Load the Madrid buildings GeoPackage and filter to only buildings with an area greater than 500 square metres. How many remain?
- Reproject the buildings GeoDataFrame to EPSG:4326 (WGS 84) and plot the result. What changes?
- Create a new column called `perimeter` containing each building's perimeter length. Plot the buildings coloured by perimeter.
- Create a `Point` geometry for a location of your choice and calculate the distance from each building's centroid to that point. Which building is closest?

## Summary

- GeoPandas extends pandas with a geometry column and CRS support, bridging tabular data analysis with spatial operations.
- Use `.loc[]` and boolean masks to index and filter rows by attribute or spatial properties.
- `to_crs()` transforms between coordinate reference systems — always use a projected CRS for distance and area calculations.
- Spatial operations like `buffer` and `distance` work element-wise across the GeoDataFrame.
- `read_file()` and `to_file()` handle common spatial formats (GeoPackage, Shapefile, GeoJSON).

Next: [Urban Analytics](5_urban.qmd) introduces `osmnx` for downloading OSM data and `momepy` for morphological analysis. The skills from this chapter are used extensively in the [Network Preparation](../recipes/networks/index.qmd) and [Statistics](../recipes/stats/index.qmd) recipes.